# Equal-Volume Cylindrical Shell Theta Baseline

This baseline uses centered cylindrical shell bands with equal shell volume.

Each shell is defined by successive outer boundaries:
- `R_i = R_max * (i / n_shells)^(1/3)`
- `Z_i = Z_max * (i / n_shells)^(1/3)`

The target is `inside_shell`, meaning an event lies inside shell band `i` and outside shell band `i-1`.


In [1]:
from __future__ import annotations

import json
import yaml
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "prepare_resum_data.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not find the XLZD repo root from the current working directory.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT / "src" / "run_cnp") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src" / "run_cnp"))
if str(REPO_ROOT / "src" / "run_mfgp") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src" / "run_mfgp"))

from cnp_clean_pipeline import load_runtime_config, predict_cnp, train_cnp
from mfgp_clean_pipeline import run_mfgp_transform_suite

PREP_CONFIG_PATH = REPO_ROOT / "xlzd_equal_volume_shell_theta" / "config" / "pipeline_config.json"
NOTEBOOK_CONFIG_PATH = REPO_ROOT / "xlzd_equal_volume_shell_theta" / "config" / "runtime_notebook_config.json"
CONFIG_PATH = REPO_ROOT / "xlzd_equal_volume_shell_theta" / "settings_equal_volume_shell_minibatch.yaml"
VALIDATION_CONFIG_PATH = REPO_ROOT / "xlzd_equal_volume_shell_theta" / "settings_equal_volume_shell_validation_minibatch.yaml"
CNP_TRAIN_CSV = REPO_ROOT / "data/out/cnp/cnp_xlzd_equal_volume_shell_v1_minibatch_output_15epochs.csv"
CNP_VALIDATION_CSV = REPO_ROOT / "data/out/cnp/cnp_xlzd_equal_volume_shell_v1_minibatch_output_validation_15epochs.csv"
DATASET_ROOT = REPO_ROOT / "outputs_equal_volume_shell_theta"

SEED = 42
STEPS_PER_EPOCH = 5000
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
REPR_DIM = 32
HIDDEN = 128
DROPOUT = 0.1
MONITOR_EVERY = 5000
MC_SAMPLES = 30
CHUNK_SIZE = 20000
GRID_POINTS = 60
RANDOM_STATE = 42
TARGET_TRANSFORMS = ["linear", "log_hf", "log_lf", "log_both"]


## 1. Choose Notebook Parameters

Edit these values here instead of changing the default JSON config.


In [2]:
NOTEBOOK_PARAMS = {
    "R_max": 1500.0,
    "Z_max": 2000.0,
    "n_shells": 100,
    "min_candidate_events": 1,
    "lf_block_size": 20000,
    "hf_block_size": 100000,
    "validation_block_size": None,
    "negative_shells": 4,
    "scale_power": 1.0/3.0
}

pd.DataFrame({
    "parameter": list(NOTEBOOK_PARAMS.keys()),
    "value": list(NOTEBOOK_PARAMS.values()),
})


,parameter,value
0,R_max,1500.000000
1,Z_max,2000.000000
2,n_shells,100.000000
3,min_candidate_events,1.000000
4,lf_block_size,20000.000000
5,hf_block_size,100000.000000
6,validation_block_size,NaN
7,negative_shells,4.000000
8,scale_power,0.333333


## Parameter Guide

- `n_shells`: higher means thinner shells and more spatial resolution; lower means thicker shells and more events per shell.
- `min_candidate_events`: higher means only well-supported shells survive; lower means more inner/sparse shells are allowed, but they will be noisier.
- `lf_block_size`: higher means fewer LF files and fewer required shell assignments; lower means more LF files and more repeated shell assignments if shell diversity is limited.
- `hf_block_size`: higher means fewer HF training files; lower means more HF training files.
- `validation_block_size`: if `None`, validation uses the HF block size; lowering it creates more validation files.
- `R_max`, `Z_max`: set the outer cylindrical boundary family used to generate the equal-volume shell bands.

This notebook writes a temporary runtime config but always reuses the same dataset folder:
- `outputs_equal_volume_shell_theta`

So rerunning the prep command overwrites the previous equal-volume shell dataset instead of creating a new dataset tree each time.


## 2. Build A Runtime Config From Those Parameters


In [3]:
prep_config = json.loads(PREP_CONFIG_PATH.read_text())
prep_config["shell"]["R_max"] = float(NOTEBOOK_PARAMS["R_max"])
prep_config["shell"]["Z_max"] = float(NOTEBOOK_PARAMS["Z_max"])
prep_config["shell"]["n_shells"] = int(NOTEBOOK_PARAMS["n_shells"])
prep_config["shell"]["min_candidate_events"] = int(NOTEBOOK_PARAMS["min_candidate_events"])
prep_config["sampling"]["lf_block_size"] = int(NOTEBOOK_PARAMS["lf_block_size"])
prep_config["sampling"]["hf_block_size"] = int(NOTEBOOK_PARAMS["hf_block_size"])
prep_config["sampling"]["validation_block_size"] = NOTEBOOK_PARAMS["validation_block_size"]
prep_config["shell"]["scale_power"] = float(NOTEBOOK_PARAMS["scale_power"])
prep_config["shell"]["negative_shells"] = int(NOTEBOOK_PARAMS["negative_shells"])

# Make sure the yamls have the correct shell geometry
for yaml_path in [CONFIG_PATH, VALIDATION_CONFIG_PATH]:
    yaml_config = yaml.safe_load(yaml_path.read_text())
    sim = yaml_config["simulation_settings"]

    sim["R_max"] = float(NOTEBOOK_PARAMS["R_max"])
    sim["Z_max"] = float(NOTEBOOK_PARAMS["Z_max"])
    sim["n_shells"] = int(NOTEBOOK_PARAMS["n_shells"])
    sim["scale_power"] = float(NOTEBOOK_PARAMS["scale_power"])

    yaml_path.write_text(
        yaml.safe_dump(
            yaml_config,
            sort_keys=False,
        )
    )

NOTEBOOK_CONFIG_PATH.write_text(json.dumps(prep_config, indent=2))

runtime = load_runtime_config(CONFIG_PATH, seed=SEED)
validation_runtime = load_runtime_config(VALIDATION_CONFIG_PATH, seed=SEED)

summary = pd.DataFrame({
    "field": [
        "version",
        "dataset_root",
        "R_max",
        "Z_max",
        "n_shells",
        "min_candidate_events",
        "lf_block_size",
        "hf_block_size",
        "validation_block_size",
        "theta_headers",
        "phi_headers",
        "target_headers",
        "scale_power",
        "negative_shells",
    ],
    "value": [
        runtime.version,
        str(DATASET_ROOT),
        prep_config["shell"].get("R_max"),
        prep_config["shell"].get("Z_max"),
        prep_config["shell"].get("n_shells"),
        prep_config["shell"].get("min_candidate_events"),
        prep_config["sampling"].get("lf_block_size"),
        prep_config["sampling"].get("hf_block_size"),
        prep_config["sampling"].get("validation_block_size"),
        prep_config["shell"].get("scale_power"),
        prep_config["shell"].get("negative_shells"),
        ", ".join(runtime.theta_headers),
        ", ".join(runtime.phi_headers),
        ", ".join(runtime.target_headers),
    ],
})
summary


,field,value
0,version,xlzd_equal_volume_shell_v1_minibatch
1,dataset_root,/home/liuser/pknauss/XLZD/outputs_equal_volume...
2,R_max,1500.0
3,Z_max,2000.0
4,n_shells,100
5,min_candidate_events,1
6,lf_block_size,20000
7,hf_block_size,100000
8,validation_block_size,None
9,theta_headers,0.333333


## 3. Prepare The Equal-Volume Shell Dataset

Run this from the repo root before training if the dataset has not been built yet.

If you want more inner-shell candidates, lower `min_candidate_events` first. If you want fewer repeated shell assignments, increase `lf_block_size`.

This prep step now clears and rebuilds `outputs_equal_volume_shell_theta`, so changing notebook parameters refreshes the same dataset instead of leaving old files behind.


In [4]:
import subprocess

cmd = [
    sys.executable,
    "xlzd_equal_volume_shell_theta/prepare_equal_volume_shell_data.py",
    "--config",
    str(NOTEBOOK_CONFIG_PATH.relative_to(REPO_ROOT)),
]

print("$ " + " ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

$ /home/liuser/miniconda3/bin/python xlzd_equal_volume_shell_theta/prepare_equal_volume_shell_data.py --config xlzd_equal_volume_shell_theta/config/runtime_notebook_config.json

[15:52:52] Loading and normalizing raw event files


Loading event files: 100%|██████████| 7/7 [00:04<00:00,  1.74it/s]


[done in 4.07s] Loaded 7 files

[15:52:56] Computing centered coordinates
[done in 0.14s] Computed z_from_center using z_center=1982.48

[15:52:56] Splitting raw events into disjoint LF/HF/validation pools
[done in 1.08s] Pool split complete

[15:52:57] Splitting pools into equal-size event blocks
[done in 0.00s] Pool block split complete

[15:52:57] Building equal-volume shell candidates from LF support
[done in 0.07s] Built 88 valid shell candidates

[15:52:58] Clearing existing dataset directory: outputs_equal_volume_shell_theta
[done in 0.15s] Removed previous equal-volume shell dataset

[15:52:58] Writing LF Training event-shell pair H5 blocks


Traceback (most recent call last):
  File "/home/liuser/pknauss/XLZD/xlzd_equal_volume_shell_theta/prepare_equal_volume_shell_data.py", line 684, in <module>
    main()
  File "/home/liuser/pknauss/XLZD/xlzd_equal_volume_shell_theta/prepare_equal_volume_shell_data.py", line 601, in main
    lf_manifest = write_h5_all_blocks(
                  ^^^^^^^^^^^^^^^^^^^^
  File "/home/liuser/pknauss/XLZD/xlzd_equal_volume_shell_theta/prepare_equal_volume_shell_data.py", line 421, in write_h5_all_blocks
    theta, phi, target, meta = build_event_pairs_per_block(
                               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/liuser/pknauss/XLZD/xlzd_equal_volume_shell_theta/prepare_equal_volume_shell_data.py", line 350, in build_event_pairs_per_block
    float(neg_shell["R_shell"]),
          ~~~~~~~~~^^^^^^^^^^^
  File "/home/liuser/miniconda3/lib/python3.12/site-packages/pandas/core/series.py", line 952, in __getitem__
    key_is_scalar = is_scalar(key)
                    ^^^^^^^^^

KeyboardInterrupt: 

## 4. Train The CNP


In [ ]:
train_result = train_cnp(
    runtime,
    steps_per_epoch=STEPS_PER_EPOCH,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    repr_dim=REPR_DIM,
    hidden=HIDDEN,
    dropout=DROPOUT,
    monitor_every=MONITOR_EVERY,
    show_monitor_plots=True,
)

pd.DataFrame({
    "artifact": ["model_path", "history_csv", "history_plot", "sample_plot"],
    "path": [
        str(train_result.model_path),
        str(train_result.history_csv),
        str(train_result.history_plot),
        str(train_result.sample_plot),
    ],
})


## 5. Predict On Training LF/HF And Validation HF


In [ ]:
train_prediction = predict_cnp(
    runtime,
    model_path=train_result.model_path,
    mc_samples=MC_SAMPLES,
    chunk_size=CHUNK_SIZE,
)

validation_prediction = predict_cnp(
    validation_runtime,
    model_path=train_result.model_path,
    mc_samples=MC_SAMPLES,
    chunk_size=CHUNK_SIZE,
)

pd.DataFrame({
    "artifact": [
        "train_csv",
        "train_heatmap",
        "train_error_heatmap",
        "validation_csv",
        "validation_heatmap",
        "validation_error_heatmap",
    ],
    "path": [
        str(train_prediction.csv_path),
        str(train_prediction.heatmap_path),
        str(train_prediction.error_heatmap_path),
        str(validation_prediction.csv_path),
        str(validation_prediction.heatmap_path),
        str(validation_prediction.error_heatmap_path),
    ],
})


## 6. Inspect CNP Outputs


In [ ]:
train_df = pd.read_csv(CNP_TRAIN_CSV)
validation_df = pd.read_csv(CNP_VALIDATION_CSV)

display(train_df.head())
display(validation_df.head())

for path in [
    train_result.history_plot,
    train_result.sample_plot,
    train_prediction.heatmap_path,
    train_prediction.error_heatmap_path,
    validation_prediction.heatmap_path,
    validation_prediction.error_heatmap_path,
]:
    print(path)
    if Path(path).exists():
        try:
            display(Image(filename=str(path)))
        except Exception:
            pass


## 7. Fit The MF-GP Transform Suite


In [ ]:
mfgp_results = run_mfgp_transform_suite(
    config_path=CONFIG_PATH,
    cnp_csv=CNP_TRAIN_CSV,
    validation_csv=CNP_VALIDATION_CSV,
    transforms=TARGET_TRANSFORMS,
    iteration=0,
    grid_points_per_axis=GRID_POINTS,
    random_state=RANDOM_STATE,
    predict_chunk_size=CHUNK_SIZE,
    verbose=True,
)

metrics_rows = []
for transform, result in mfgp_results.items():
    metrics_path = getattr(result, "metrics_json", None)
    if metrics_path and Path(metrics_path).exists():
        payload = json.loads(Path(metrics_path).read_text())
        metrics_rows.append({"transform": transform, **payload})
pd.DataFrame(metrics_rows) if metrics_rows else pd.DataFrame()


## 8. Display MF-GP Plots


In [ ]:
EXPERIMENT_TITLES = {
    "linear": "Normal MF-GP",
    "log_hf": "Log HF: emulate log10(y_raw)",
    "log_lf": "Log LF: use log10(y_cnp)",
    "log_both": "Log HF + Log LF: use log10(y_raw) and log10(y_cnp)",
}

PLOT_SELECTIONS = {
    "linear": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: y with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation 3-sigma: log10(y) with linear sigma", "validation_across_theta_log_linear_sigma_plot"),
        ("Validation parity", "validation_parity_linear_plot"),
    ],
    "log_hf": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: log10 target with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation 3-sigma: y with 10**(mu±sigma)", "validation_across_theta_log_plot"),
        ("Validation parity", "validation_parity_log_plot"),
    ],
    "log_lf": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: y with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation 3-sigma: log10(y) with linear sigma", "validation_across_theta_log_linear_sigma_plot"),
        ("Validation parity", "validation_parity_linear_plot"),
    ],
    "log_both": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: log10 target with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation 3-sigma: y with 10**(mu±sigma)", "validation_across_theta_log_plot"),
        ("Validation parity", "validation_parity_log_plot"),
    ],
}

def show_plot(result, label, key):
    value = getattr(result, key, None)
    if not value:
        return
    print(f"{label}: {value}")
    path = Path(value)
    if path.suffix.lower() == ".png" and path.exists():
        try:
            display(Image(filename=str(path)))
        except Exception:
            pass

for transform in TARGET_TRANSFORMS:
    result = mfgp_results.get(transform)
    if result is None:
        continue
    print(f"=== {EXPERIMENT_TITLES[transform]} ===")
    for label, key in PLOT_SELECTIONS[transform]:
        show_plot(result, label, key)
